In [1]:
from pyspark.context import SparkContext
from pyspark.sql import SparkSession

In [2]:
import pyspark.sql.types as tp
from pyspark.sql import functions as F

In [3]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from pyspark.sql.types import *

from pyspark.ml.feature import StringIndexer
from pyspark.ml.feature import OneHotEncoder
from pyspark.ml.feature import VectorAssembler

from pyspark.ml.regression import LinearRegression
from pyspark.ml.regression import DecisionTreeRegressor
from pyspark.ml.regression import RandomForestRegressor

from pyspark.ml.evaluation import RegressionEvaluator

from pyspark.ml.tuning import ParamGridBuilder, CrossValidator

from pyspark.ml import Pipeline

In [4]:
spark = SparkSession.builder \
    .appName("Spark ML Assignment") \
    .getOrCreate()

In [9]:
df = spark.read.csv(
    r"C:\Users\likhi\Downloads\MASTERING APACHE SPARK USING PYTHON\Spark_ML_Assignment\train.csv",
    header=True,
    inferSchema=True
)

In [10]:
df.show(5)

+-------+----------+------+----+----------+-------------+--------------------------+--------------+------------------+------------------+------------------+--------+
|User_ID|Product_ID|Gender| Age|Occupation|City_Category|Stay_In_Current_City_Years|Marital_Status|Product_Category_1|Product_Category_2|Product_Category_3|Purchase|
+-------+----------+------+----+----------+-------------+--------------------------+--------------+------------------+------------------+------------------+--------+
|1000001| P00069042|     F|0-17|        10|            A|                         2|             0|                 3|              NULL|              NULL|    8370|
|1000001| P00248942|     F|0-17|        10|            A|                         2|             0|                 1|                 6|                14|   15200|
|1000001| P00087842|     F|0-17|        10|            A|                         2|             0|                12|              NULL|              NULL|    1422|
|100

In [11]:
df.printSchema()

root
 |-- User_ID: integer (nullable = true)
 |-- Product_ID: string (nullable = true)
 |-- Gender: string (nullable = true)
 |-- Age: string (nullable = true)
 |-- Occupation: integer (nullable = true)
 |-- City_Category: string (nullable = true)
 |-- Stay_In_Current_City_Years: string (nullable = true)
 |-- Marital_Status: integer (nullable = true)
 |-- Product_Category_1: integer (nullable = true)
 |-- Product_Category_2: integer (nullable = true)
 |-- Product_Category_3: integer (nullable = true)
 |-- Purchase: integer (nullable = true)



In [12]:
df.show(10, truncate=False)

+-------+----------+------+-----+----------+-------------+--------------------------+--------------+------------------+------------------+------------------+--------+
|User_ID|Product_ID|Gender|Age  |Occupation|City_Category|Stay_In_Current_City_Years|Marital_Status|Product_Category_1|Product_Category_2|Product_Category_3|Purchase|
+-------+----------+------+-----+----------+-------------+--------------------------+--------------+------------------+------------------+------------------+--------+
|1000001|P00069042 |F     |0-17 |10        |A            |2                         |0             |3                 |NULL              |NULL              |8370    |
|1000001|P00248942 |F     |0-17 |10        |A            |2                         |0             |1                 |6                 |14                |15200   |
|1000001|P00087842 |F     |0-17 |10        |A            |2                         |0             |12                |NULL              |NULL              |1422    

In [13]:
df.describe().show()

+-------+------------------+----------+------+------+-----------------+-------------+--------------------------+-------------------+------------------+------------------+------------------+-----------------+
|summary|           User_ID|Product_ID|Gender|   Age|       Occupation|City_Category|Stay_In_Current_City_Years|     Marital_Status|Product_Category_1|Product_Category_2|Product_Category_3|         Purchase|
+-------+------------------+----------+------+------+-----------------+-------------+--------------------------+-------------------+------------------+------------------+------------------+-----------------+
|  count|            550068|    550068|550068|550068|           550068|       550068|                    550068|             550068|            550068|            376430|            166821|           550068|
|   mean|1003028.8424013031|      NULL|  NULL|  NULL|8.076706879876669|         NULL|         1.468494139793958|0.40965298835780306| 5.404270017525106| 9.84232925112238

In [14]:
print("Duplicates :", df.count() - df.dropDuplicates().count())

Duplicates : 0


In [15]:
df = df.dropDuplicates()

In [17]:
# Find the average Purchase amount

from pyspark.sql.functions import avg

df.select(avg("Purchase").alias("Average_Purchase")).show()

+-----------------+
| Average_Purchase|
+-----------------+
|9263.968712959126|
+-----------------+



In [18]:
#Count Null Values
from pyspark.sql.functions import col, when, count

df.select([
    count(when(col(c).isNull(), c)).alias(c)
    for c in df.columns
]).show()

+-------+----------+------+---+----------+-------------+--------------------------+--------------+------------------+------------------+------------------+--------+
|User_ID|Product_ID|Gender|Age|Occupation|City_Category|Stay_In_Current_City_Years|Marital_Status|Product_Category_1|Product_Category_2|Product_Category_3|Purchase|
+-------+----------+------+---+----------+-------------+--------------------------+--------------+------------------+------------------+------------------+--------+
|      0|         0|     0|  0|         0|            0|                         0|             0|                 0|            173638|            383247|       0|
+-------+----------+------+---+----------+-------------+--------------------------+--------------+------------------+------------------+------------------+--------+



In [21]:
#Remove Null Values
df.dropna()

DataFrame[User_ID: int, Product_ID: string, Gender: string, Age: string, Occupation: int, City_Category: string, Stay_In_Current_City_Years: string, Marital_Status: int, Product_Category_1: int, Product_Category_2: int, Product_Category_3: int, Purchase: int]

In [22]:
df.select([
    count(when(col(c).isNull(), c)).alias(c)
    for c in df.columns
]).show()

+-------+----------+------+---+----------+-------------+--------------------------+--------------+------------------+------------------+------------------+--------+
|User_ID|Product_ID|Gender|Age|Occupation|City_Category|Stay_In_Current_City_Years|Marital_Status|Product_Category_1|Product_Category_2|Product_Category_3|Purchase|
+-------+----------+------+---+----------+-------------+--------------------------+--------------+------------------+------------------+------------------+--------+
|      0|         0|     0|  0|         0|            0|                         0|             0|                 0|            173638|            383247|       0|
+-------+----------+------+---+----------+-------------+--------------------------+--------------+------------------+------------------+------------------+--------+



In [23]:
#Count Distinct Values

from pyspark.sql.functions import countDistinct

df.select([
    countDistinct(c).alias(c)
    for c in df.columns
]).show()

+-------+----------+------+---+----------+-------------+--------------------------+--------------+------------------+------------------+------------------+--------+
|User_ID|Product_ID|Gender|Age|Occupation|City_Category|Stay_In_Current_City_Years|Marital_Status|Product_Category_1|Product_Category_2|Product_Category_3|Purchase|
+-------+----------+------+---+----------+-------------+--------------------------+--------------+------------------+------------------+------------------+--------+
|   5891|      3631|     2|  7|        21|            3|                         5|             2|                20|                17|                15|   18105|
+-------+----------+------+---+----------+-------------+--------------------------+--------------+------------------+------------------+------------------+--------+



In [24]:
# Count Category Values
df.groupBy("Gender").count().show()


+------+------+
|Gender| count|
+------+------+
|     F|135809|
|     M|414259|
+------+------+



In [25]:
df.groupBy("Age").count().show()

+-----+------+
|  Age| count|
+-----+------+
|18-25| 99660|
|26-35|219587|
| 0-17| 15102|
|46-50| 45701|
|51-55| 38501|
|36-45|110013|
|  55+| 21504|
+-----+------+



In [26]:
df.groupBy("Occupation").count().show()

+----------+-----+
|Occupation|count|
+----------+-----+
|        12|31179|
|         1|47426|
|        13| 7728|
|        16|25371|
|         6|20355|
|         3|17650|
|        20|33562|
|         5|12177|
|        19| 8461|
|        15|12165|
|        17|40043|
|         9| 6291|
|         4|72308|
|         8| 1546|
|         7|59133|
|        10|12930|
|        11|11586|
|        14|27309|
|         2|26588|
|         0|69638|
+----------+-----+
only showing top 20 rows


In [27]:
df.groupBy("City_Category").count().show()

+-------------+------+
|City_Category| count|
+-------------+------+
|            B|231173|
|            C|171175|
|            A|147720|
+-------------+------+



In [28]:
df.groupBy("Stay_In_Current_City_Years").count().show()

+--------------------------+------+
|Stay_In_Current_City_Years| count|
+--------------------------+------+
|                         3| 95285|
|                         0| 74398|
|                        4+| 84726|
|                         1|193821|
|                         2|101838|
+--------------------------+------+



In [29]:
df.groupBy("Marital_Status").count().show()

+--------------+------+
|Marital_Status| count|
+--------------+------+
|             1|225337|
|             0|324731|
+--------------+------+



##### Calculate average Purchase

In [30]:
from pyspark.sql.functions import avg

df.groupBy("Gender") \
  .agg(avg("Purchase").alias("Average_Purchase")) \
  .show()

+------+-----------------+
|Gender| Average_Purchase|
+------+-----------------+
|     F|8734.565765155476|
|     M|9437.526040472265|
+------+-----------------+



In [31]:
df.groupBy("Age") \
  .agg(avg("Purchase").alias("Average_Purchase")) \
  .orderBy("Age") \
  .show()

+-----+-----------------+
|  Age| Average_Purchase|
+-----+-----------------+
| 0-17|8933.464640444974|
|18-25|9169.663606261289|
|26-35|9252.690632869888|
|36-45|9331.350694917874|
|46-50|9208.625697468327|
|51-55|9534.808030960236|
|  55+|9336.280459449405|
+-----+-----------------+



In [32]:
df.groupBy("City_Category") \
  .agg(avg("Purchase").alias("Average_Purchase")) \
  .show()

+-------------+-----------------+
|City_Category| Average_Purchase|
+-------------+-----------------+
|            B|9151.300562781986|
|            C| 9719.92099313568|
|            A|8911.939216084484|
+-------------+-----------------+



In [33]:
df.groupBy("Stay_In_Current_City_Years") \
  .agg(avg("Purchase").alias("Average_Purchase")) \
  .orderBy("Stay_In_Current_City_Years") \
  .show()

+--------------------------+-----------------+
|Stay_In_Current_City_Years| Average_Purchase|
+--------------------------+-----------------+
|                         0|9180.075122987177|
|                         1|9250.145923300364|
|                         2|9320.429810090536|
|                         3|9286.904119221284|
|                        4+| 9275.59887165687|
+--------------------------+-----------------+



In [34]:
df.groupBy("Marital_Status") \
  .agg(avg("Purchase").alias("Average_Purchase")) \
  .show()

+--------------+-----------------+
|Marital_Status| Average_Purchase|
+--------------+-----------------+
|             1|9261.174574082374|
|             0|9265.907618921507|
+--------------+-----------------+



##### Label Encoding

In [35]:
from pyspark.ml.feature import StringIndexer

In [36]:
age_indexer = StringIndexer(
    inputCol="Age",
    outputCol="Age_Index",
    handleInvalid="keep"
)

In [37]:
gender_indexer = StringIndexer(
    inputCol="Gender",
    outputCol="Gender_Index",
    handleInvalid="keep"
)

In [38]:
stay_indexer = StringIndexer(
    inputCol="Stay_In_Current_City_Years",
    outputCol="Stay_Index",
    handleInvalid="keep"
)

In [39]:
city_indexer = StringIndexer(
    inputCol="City_Category",
    outputCol="City_Index",
    handleInvalid="keep"
)

##### OneHotEncoding

In [40]:
from pyspark.ml.feature import OneHotEncoder

In [41]:
occupation_indexer = StringIndexer(
    inputCol="Occupation",
    outputCol="Occupation_Index",
    handleInvalid="keep"
)

occupation_model = occupation_indexer.fit(df)

df = occupation_model.transform(df)

In [44]:
gender_model = gender_indexer.fit(df)
df = gender_model.transform(df)

city_model = city_indexer.fit(df)
df = city_model.transform(df)

In [45]:
encoder_model = encoder.fit(df)

In [46]:
df = encoder_model.transform(df)

In [47]:
df.select(
    "Gender",
    "Gender_Index",
    "Gender_OHE",
    "City_Category",
    "City_Index",
    "City_OHE",
    "Occupation",
    "Occupation_Index",
    "Occupation_OHE"
).show(truncate=False)

+------+------------+-------------+-------------+----------+-------------+----------+----------------+---------------+
|Gender|Gender_Index|Gender_OHE   |City_Category|City_Index|City_OHE     |Occupation|Occupation_Index|Occupation_OHE |
+------+------------+-------------+-------------+----------+-------------+----------+----------------+---------------+
|M     |0.0         |(2,[0],[1.0])|B            |0.0       |(3,[0],[1.0])|0         |1.0             |(21,[1],[1.0]) |
|F     |1.0         |(2,[1],[1.0])|C            |1.0       |(3,[1],[1.0])|1         |3.0             |(21,[3],[1.0]) |
|M     |0.0         |(2,[0],[1.0])|B            |0.0       |(3,[0],[1.0])|4         |0.0             |(21,[0],[1.0]) |
|M     |0.0         |(2,[0],[1.0])|B            |0.0       |(3,[0],[1.0])|4         |0.0             |(21,[0],[1.0]) |
|M     |0.0         |(2,[0],[1.0])|A            |2.0       |(3,[2],[1.0])|17        |4.0             |(21,[4],[1.0]) |
|M     |0.0         |(2,[0],[1.0])|A            

In [48]:
df.printSchema()

root
 |-- User_ID: integer (nullable = true)
 |-- Product_ID: string (nullable = true)
 |-- Gender: string (nullable = true)
 |-- Age: string (nullable = true)
 |-- Occupation: integer (nullable = true)
 |-- City_Category: string (nullable = true)
 |-- Stay_In_Current_City_Years: string (nullable = true)
 |-- Marital_Status: integer (nullable = true)
 |-- Product_Category_1: integer (nullable = true)
 |-- Product_Category_2: integer (nullable = true)
 |-- Product_Category_3: integer (nullable = true)
 |-- Purchase: integer (nullable = true)
 |-- Occupation_Index: double (nullable = false)
 |-- Gender_Index: double (nullable = false)
 |-- City_Index: double (nullable = false)
 |-- Gender_OHE: vector (nullable = true)
 |-- City_OHE: vector (nullable = true)
 |-- Occupation_OHE: vector (nullable = true)



In [49]:
feature_columns = [

    "Gender_OHE",

    "Age_Index",

    "Occupation_OHE",

    "City_OHE",

    "Stay_Index",

    "Marital_Status",

    "Product_Category_1",

    "Product_Category_2",

    "Product_Category_3"

]

In [50]:
from pyspark.ml.feature import VectorAssembler

In [51]:
assembler = VectorAssembler(

    inputCols=feature_columns,

    outputCol="features"

)

In [54]:
age_model = age_indexer.fit(df)
df = age_model.transform(df)